In [19]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from collections import Counter
import albumentations as A
import shutil

In [20]:

# Paths
input_img_dir = r"D:\FYP\Datasets\Dataset with good_defect mix\YOLO_split\train\images_noaug"
input_lbl_dir = r"D:\FYP\Datasets\Dataset with good_defect mix\YOLO_split\train\labels_noaug"
output_img_dir = r"D:\FYP\Datasets\Dataset with good_defect mix\YOLO_split\train\images"
output_lbl_dir = r"D:\FYP\Datasets\Dataset with good_defect mix\YOLO_split\train\labels"

os.makedirs(output_img_dir, exist_ok=True)
os.makedirs(output_lbl_dir, exist_ok=True)


In [ ]:

# Step 1 – Count occurrences of each class
all_labels = []
label_files = [f for f in os.listdir(input_lbl_dir) if f.endswith(".txt")]

for lbl_file in label_files:
    lbl_path = os.path.join(input_lbl_dir, lbl_file)
    with open(lbl_path, "r") as f:
        lines = f.readlines()
        for line in lines:
            cls_id = int(line.strip().split()[0])
            all_labels.append(cls_id)

counter = Counter(all_labels)
print("Original class distribution:", counter)

# Step 2 – Identify minority classes
threshold = min(counter.values()) * 1.5  # define how much below max count is considered minority
minority_classes = [cls for cls, count in counter.items() if count < threshold]
print("Minority classes:", minority_classes)


In [ ]:

# Step 3 – Define augmentation transform
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    A.Rotate(limit=10, p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0, p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


In [ ]:

# Step 4 – Copy original dataset
print("Copying original dataset...")
for lbl_file in tqdm(label_files):
    base_name = lbl_file.rsplit(".", 1)[0]
    img_file = base_name + ".jpg"
    shutil.copy(os.path.join(input_img_dir, img_file), os.path.join(output_img_dir, img_file))
    shutil.copy(os.path.join(input_lbl_dir, lbl_file), os.path.join(output_lbl_dir, lbl_file))

# Step 5 – Augment and oversample minority class images
dup_count = 3  # number of augmented copies per minority image

print("Augmenting minority class images...")
for lbl_file in tqdm(label_files):
    lbl_path = os.path.join(input_lbl_dir, lbl_file)
    with open(lbl_path, "r") as f:
        lines = f.readlines()
        classes_in_image = {int(line.strip().split()[0]) for line in lines}

    if any(cls in minority_classes for cls in classes_in_image):
        base_name = lbl_file.rsplit(".", 1)[0]
        img_file = base_name + ".jpg"
        img_path = os.path.join(input_img_dir, img_file)
        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        # Parse bounding boxes
        bboxes = []
        class_labels = []
        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            bboxes.append([cx, cy, bw, bh])
            class_labels.append(cls_id)

        for i in range(dup_count):
            transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
            aug_img = transformed['image']
            aug_bboxes = transformed['bboxes']
            aug_labels = transformed['class_labels']

            if len(aug_bboxes) == 0:
                continue

            # Save augmented image
            out_img_file = base_name + f"_aug{i}.jpg"
            out_img_path = os.path.join(output_img_dir, out_img_file)
            cv2.imwrite(out_img_path, aug_img)

            # Save augmented labels
            out_lbl_file = base_name + f"_aug{i}.txt"
            out_lbl_path = os.path.join(output_lbl_dir, out_lbl_file)
            with open(out_lbl_path, "w") as f_out:
                for cls_id, box in zip(aug_labels, aug_bboxes):
                    cx, cy, bw, bh = box
                    line = f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n"
                    f_out.write(line)

print("Class balancing with augmentation completed. Files saved in 'balanced_augmented_dataset/'")


In [14]:
images_dir = r"D:\FYP\Datasets\NEW\train\images_aug2"
labels_dir = r"D:\FYP\Datasets\NEW\train\labels_aug2"

aug_images_dir = r"D:\FYP\Datasets\NEW\train\images"
aug_labels_dir = r"D:\FYP\Datasets\NEW\train\labels"

In [3]:
"""
Copy originals -> then augment and add augmented files into same output folders.
Update the top CONFIG paths before running.
"""

import os
import shutil
import random
import cv2
import yaml
import albumentations as A
from collections import defaultdict

# ---------------------------
# CONFIG - update these paths
# ---------------------------
images_dir = r"D:\FYP\Datasets\trail\YOLO_split\train\images_pre"   # original images (source)
labels_dir = r"D:\FYP\Datasets\trail\YOLO_split\train\labels"   # original labels (source)

out_images_dir = r"D:\FYP\Datasets\trail\YOLO_split\train\images_aug"   # destination (will contain originals + augmented)
out_labels_dir = r"D:\FYP\Datasets\trail\YOLO_split\train\labels_pre"    # destination (will contain originals + augmented)

dataset_yaml = r"D:\FYP\Datasets\trail\YOLO_split\dataset.yaml"

os.makedirs(out_images_dir, exist_ok=True)
os.makedirs(out_labels_dir, exist_ok=True)

bbox_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    A.Rotate(limit=10, p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0, p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.1))

image_only_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.5),
    A.Rotate(limit=15, p=0.5),
])

def find_image_file(base_name, search_dir):
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(search_dir, base_name + ext)
        if os.path.exists(p):
            return p
    return None

def read_yolo_labels(label_path):
    out = []
    with open(label_path, "r") as f:
        for ln in f:
            parts = ln.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:5])
                out.append((cls, x, y, w, h))
            except:
                continue
    return out

def write_yolo_labels(path, boxes):
    with open(path, "w") as f:
        for (cls, x, y, w, h) in boxes:
            f.write(f"{int(cls)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

def pre_clamp_yolo_box(x, y, w, h, eps=1e-6):
    w = min(max(w, 0.0), 1.0 - eps)
    h = min(max(h, 0.0), 1.0 - eps)
    half_w = w / 2.0
    half_h = h / 2.0
    x = min(max(x, half_w + eps), 1.0 - half_w - eps)
    y = min(max(y, half_h + eps), 1.0 - half_h - eps)
    return x, y, w, h

def clamp_yolo_box(box):
    x, y, w, h = box
    x = min(max(x, 0.0), 1.0)
    y = min(max(y, 0.0), 1.0)
    w = min(max(w, 0.0), 1.0)
    h = min(max(h, 0.0), 1.0)
    return [x, y, w, h]

def safe_copy_file(src, dst_folder):
    base = os.path.basename(src)
    dst = os.path.join(dst_folder, base)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        return dst
    # if exists, create a non-colliding name
    name, ext = os.path.splitext(base)
    i = 1
    while True:
        new_name = f"{name}_orig{i}{ext}"
        dst = os.path.join(dst_folder, new_name)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            return dst
        i += 1

def copy_original_dataset(images_dir, labels_dir, out_images_dir, out_labels_dir):
    print("Copying original images...")
    # copy images
    for fname in os.listdir(images_dir):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        src = os.path.join(images_dir, fname)
        safe_copy_file(src, out_images_dir)
    print("Copying original labels...")
    # copy labels
    for fname in os.listdir(labels_dir):
        if not fname.lower().endswith(".txt"):
            continue
        src = os.path.join(labels_dir, fname)
        safe_copy_file(src, out_labels_dir)
    print("Copy complete. Originals now in output folders (no overwrite).")

def analyze_labels_counts(labels_folder):
    counts = defaultdict(int)
    label_files = [f for f in os.listdir(labels_folder) if f.endswith(".txt")]
    for lf in label_files:
        path = os.path.join(labels_folder, lf)
        boxes = read_yolo_labels(path)
        for cls, x, y, w, h in boxes:
            counts[int(cls)] += 1
    return dict(counts)

def detect_good_class(labels_dir, dataset_yaml=None):
    good_id = None
    class_names = None
    if dataset_yaml and os.path.exists(dataset_yaml):
        try:
            with open(dataset_yaml, "r") as f:
                data = yaml.safe_load(f)
            if "names" in data:
                class_names = data["names"]
                if "good" in class_names:
                    good_id = class_names.index("good")
                    print(f"Detected 'good' from YAML -> id {good_id}")
                    return good_id
        except Exception:
            pass

    # fallback heuristic: choose class with largest average bbox area (w*h) or smallest count if uncertain
    cls_counts = defaultdict(int)
    cls_area = defaultdict(float)
    files = [f for f in os.listdir(labels_dir) if f.endswith(".txt")]
    for lf in files:
        boxes = read_yolo_labels(os.path.join(labels_dir, lf))
        for cls, x, y, w, h in boxes:
            cls_counts[int(cls)] += 1
            cls_area[int(cls)] += (w*h)
    if not cls_counts:
        raise RuntimeError("No labels found to detect good class.")

    cls_avg_area = {k: cls_area[k] / cls_counts[k] for k in cls_counts}
    # candidate with largest avg area
    candidate, area = max(cls_avg_area.items(), key=lambda x: x[1])
    if area >= 0.70:
        print(f"Auto-detected 'good' by avg area -> class {candidate} (area={area:.3f})")
        return candidate
    # otherwise pick smallest-count class as fallback
    candidate2 = min(cls_counts.items(), key=lambda x: x[1])[0]
    print(f"Fallback: selecting class {candidate2} as 'good' (could be wrong).")
    return candidate2

def augment_bbox_image(img_path, label_path, out_img_name, out_lbl_name, out_images_dir, out_labels_dir):
    image = cv2.imread(img_path)
    if image is None:
        print("⚠️ Could not read image:", img_path)
        return False
    raw = read_yolo_labels(label_path)
    bboxes = []
    class_labels = []
    for cls, x, y, w, h in raw:
        x2,y2,w2,h2 = pre_clamp_yolo_box(x,y,w,h)
        bboxes.append([x2,y2,w2,h2])
        class_labels.append(int(cls))
    if not bboxes:
        return False
    try:
        aug = bbox_transform(image=image, bboxes=bboxes, class_labels=class_labels)
    except Exception as e:
        # retry with stronger clamp
        # print("Albumentations error, retry with stronger clamp:", e)
        bboxes2 = []
        class_labels2 = []
        for cls,x,y,w,h in raw:
            x2,y2,w2,h2 = pre_clamp_yolo_box(x,y,w,h, eps=1e-4)
            bboxes2.append([x2,y2,w2,h2])
            class_labels2.append(int(cls))
        aug = bbox_transform(image=image, bboxes=bboxes2, class_labels=class_labels2)

    aug_img = aug["image"]
    aug_bboxes = aug.get("bboxes", [])
    aug_classes = aug.get("class_labels", [])
    final_boxes = []
    for c,b in zip(aug_classes, aug_bboxes):
        bx = clamp_yolo_box(b)
        if bx[2] <= 1e-4 or bx[3] <= 1e-4:
            continue
        final_boxes.append((int(c), bx[0], bx[1], bx[2], bx[3]))
    cv2.imwrite(os.path.join(out_images_dir, out_img_name), aug_img)
    write_yolo_labels(os.path.join(out_labels_dir, out_lbl_name), final_boxes)
    return True

def augment_good_image(img_path, out_img_name, out_lbl_name, cls, out_images_dir, out_labels_dir):
    image = cv2.imread(img_path)
    if image is None:
        print("⚠️ Could not read image:", img_path)
        return False
    aug_img = image_only_transform(image=image)["image"]
    full_box = (int(cls), 0.5, 0.5, 1.0, 1.0)
    cv2.imwrite(os.path.join(out_images_dir, out_img_name), aug_img)
    write_yolo_labels(os.path.join(out_labels_dir, out_lbl_name), [full_box])
    return True

if __name__ == "__main__":
    # 1) Copy original dataset into output folder (safe copy)
    copy_original_dataset(images_dir, labels_dir, out_images_dir, out_labels_dir)

    # 2) Analyze original counts (before adding aug) — based on the original labels_dir
    print("\nCounts from original label folder (source):")
    original_counts = analyze_labels_counts(labels_dir)
    print(original_counts)

    # 3) Detect good class id
    good_class_id = detect_good_class(labels_dir, dataset_yaml)

    # 4) Prepare label-files grouped by class (source)
    all_label_files = [f for f in os.listdir(labels_dir) if f.endswith(".txt")]
    files_by_class = defaultdict(list)
    for lf in all_label_files:
        boxes = read_yolo_labels(os.path.join(labels_dir, lf))
        seen = set()
        for cls, x, y, w, h in boxes:
            if lf not in seen:
                files_by_class[int(cls)].append(lf)
                seen.add(lf)

    if not files_by_class:
        raise RuntimeError("No labeled files found in labels_dir.")

    # 5) Determine augmentation target: the maximum bounding box count among classes (original)
    max_count = max(original_counts.values())
    print(f"\nTarget (per-class bbox count) = {max_count}")

    random.seed(42)
    for cls, cnt in original_counts.items():
        deficit = max_count - cnt
        if deficit <= 0:
            print(f"Class {cls}: {cnt} boxes (no augmentation needed).")
            continue
        print(f"\nAugmenting class {cls}: need {deficit} more boxes")

        if cls not in files_by_class or len(files_by_class[cls]) == 0:
            print(f"WARNING: no source files for class {cls}; skipping.")
            continue

        for i in range(deficit):
            src_label = random.choice(files_by_class[cls])
            base = os.path.splitext(src_label)[0]
            src_img = find_image_file(base, images_dir)
            if src_img is None:
                print(f"⚠️ Source image for {src_label} not found, skipping.")
                continue

            out_img_name = f"{base}_aug_{cls}_{i}.jpg"
            out_lbl_name = f"{base}_aug_{cls}_{i}.txt"

            if cls == good_class_id:
                augment_good_image(src_img, out_img_name, out_lbl_name, cls, out_images_dir, out_labels_dir)
            else:
                augment_bbox_image(src_img, os.path.join(labels_dir, src_label), out_img_name, out_lbl_name, out_images_dir, out_labels_dir)

    # 7) Final recount on output labels folder (originals + augmented)
    print("\nFinal counts in output labels folder (originals + augmented):")
    final_counts = analyze_labels_counts(out_labels_dir)
    print(final_counts)

    print("\nFinished. Originals copied to:", out_images_dir, out_labels_dir)
    print("Augmented files added into the same output folders.")


C:\Users\Akiellan\AppData\Roaming\Python\Python311\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Copying original images...
Copying original labels...
Copy complete. Originals now in output folders (no overwrite).

Counts from original label folder (source):
{1: 996, 2: 2344, 0: 809, 3: 3999}
Detected 'good' from YAML -> id 3

Target (per-class bbox count) = 3999

Augmenting class 1: need 3003 more boxes

Augmenting class 2: need 1655 more boxes

Augmenting class 0: need 3190 more boxes
Class 3: 3999 boxes (no augmentation needed).

Final counts in output labels folder (originals + augmented):
{1: 5229, 2: 8162, 0: 5099, 3: 3999}

Finished. Originals copied to: D:\FYP\Datasets\trail\YOLO_split\train\images_aug D:\FYP\Datasets\trail\YOLO_split\train\labels_pre
Augmented files added into the same output folders.


In [2]:
"""
generate_yolo_yaml.py

Usage:
  - Update CONFIG section (paths).
  - Run: python generate_yolo_yaml.py
Output:
  - dataset_yaml_path (YAML file for YOLO)
  - optional train.txt / val.txt lists if split created
"""

import os
import yaml
import glob
import random

# ----------------- CONFIG -----------------
# Source folders containing your images and labels (YOLO .txt)
images_dir = r"D:\FYP\Datasets\trail\YOLO_split\train\images_pre"   # folder with images (all)
labels_dir = r"D:\FYP\Datasets\trail\YOLO_split\train\labels"  # folder with .txt labels (YOLO)

# Optionally provide a classes file (one name per line) or dataset yaml to copy names from
classes_txt = None  # e.g. r"D:\FYP\Datasets\NEW\classes.txt" or None
existing_yaml = None  # e.g. r"D:\FYP\Datasets\NEW\dataset.yaml" or None

# Output YAML path
dataset_yaml_path = r"D:\FYP\Datasets\trail\dataset.yaml"

# If val_images_dir is provided, script will use it.
# If None, script will create train.txt and val.txt lists (in same folder as dataset_yaml_path) using split_ratio.
val_images_dir = None  # or "D:\\...\\val\\images" (if you already have a validation folder)

# If val_images_dir is None, split ratio (train / val)
split_ratio = 0.80  # fraction of images used for training

# Option: whether to write absolute paths into lists/YAML (True) or relative (False)
write_absolute_paths = True

# ----------------- helpers -----------------
def read_classes_from_txt(path):
    if not path or not os.path.exists(path):
        return None
    names = []
    with open(path, "r", encoding="utf8") as f:
        for ln in f:
            name = ln.strip()
            if name:
                names.append(name)
    return names if names else None

def read_classes_from_yaml(path):
    if not path or not os.path.exists(path):
        return None
    try:
        with open(path, "r", encoding="utf8") as f:
            data = yaml.safe_load(f)
        names = data.get("names") or data.get("classes") or None
        if isinstance(names, dict):
            # convert dict to sorted list by key
            names_list = [names[k] for k in sorted(names, key=lambda x: int(x))]
            return names_list
        return names
    except Exception:
        return None

def collect_label_class_ids(labels_dir):
    class_ids = set()
    txt_files = glob.glob(os.path.join(labels_dir, "*.txt"))
    for tf in txt_files:
        with open(tf, "r", encoding="utf8") as f:
            for ln in f:
                parts = ln.strip().split()
                if len(parts) < 1:
                    continue
                try:
                    cls = int(float(parts[0]))
                except Exception:
                    continue
                class_ids.add(cls)
    return sorted(class_ids)

def collect_image_files(images_dir):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(images_dir, e)))
    files = sorted(files)
    return files

def write_list(path, items):
    with open(path, "w", encoding="utf8") as f:
        for it in items:
            f.write(it + "\n")

# ----------------- main -----------------
if __name__ == "__main__":
    # 1) detect class ids from labels
    if not os.path.isdir(labels_dir):
        raise SystemExit(f"Labels folder not found: {labels_dir}")
    class_ids = collect_label_class_ids(labels_dir)
    if not class_ids:
        raise SystemExit("No class ids found in labels folder. Are labels in YOLO format?")

    # 2) try to load names from provided sources
    names = None
    if classes_txt:
        names = read_classes_from_txt(classes_txt)
    if names is None and existing_yaml:
        names = read_classes_from_yaml(existing_yaml)
    if names is None:
        # try to find dataset.yaml or data.yaml in parent folders
        cand = os.path.join(os.path.dirname(labels_dir), "dataset.yaml")
        if os.path.exists(cand):
            names = read_classes_from_yaml(cand)
        else:
            cand2 = os.path.join(os.path.dirname(labels_dir), "data.yaml")
            if os.path.exists(cand2):
                names = read_classes_from_yaml(cand2)

    # 3) build default names if we couldn't read any
    max_id = max(class_ids)
    nc = max_id + 1
    if names:
        # if names length not match nc, warn and fallback to placeholder names for missing
        if len(names) < nc:
            print("Warning: loaded names length < number of classes detected. Filling missing with placeholders.")
            for i in range(nc - len(names)):
                names.append(f"class_{len(names)}")
        elif len(names) > nc:
            # truncate or keep? We'll truncate to nc but print a warning
            print("Warning: loaded names length > detected class ids. Truncating names to match highest class id.")
            names = names[:nc]
    else:
        names = [f"class_{i}" for i in range(nc)]
        print("No class names file found. Created placeholder names:", names)

    # 4) prepare train/val references
    dataset_yaml_dir = os.path.dirname(dataset_yaml_path) or "."
    os.makedirs(dataset_yaml_dir, exist_ok=True)

    if val_images_dir:
        # user supplied a separate val folder - we will point yaml to folders
        train_path = images_dir if write_absolute_paths else os.path.relpath(images_dir, dataset_yaml_dir)
        val_path = val_images_dir if write_absolute_paths else os.path.relpath(val_images_dir, dataset_yaml_dir)
    else:
        # create train.txt and val.txt lists from images_dir
        images = collect_image_files(images_dir)
        if not images:
            raise SystemExit("No image files found in images_dir.")
        random.seed(42)
        random.shuffle(images)
        n = len(images)
        n_train = int(n * split_ratio)
        train_images = images[:n_train]
        val_images = images[n_train:]

        # convert to absolute or relative paths
        if not write_absolute_paths:
            train_images = [os.path.relpath(p, dataset_yaml_dir) for p in train_images]
            val_images = [os.path.relpath(p, dataset_yaml_dir) for p in val_images]

        train_txt = os.path.join(dataset_yaml_dir, "train.txt")
        val_txt = os.path.join(dataset_yaml_dir, "val.txt")
        write_list(train_txt, train_images)
        write_list(val_txt, val_images)
        print(f"Wrote split files: {train_txt} ({len(train_images)} lines), {val_txt} ({len(val_images)} lines)")

        train_path = train_txt
        val_path = val_txt

    # 5) assemble YAML dict
    yaml_dict = {
        "train": str(train_path),
        "val": str(val_path),
        "nc": nc,
        "names": names
    }

    # 6) write YAML
    with open(dataset_yaml_path, "w", encoding="utf8") as f:
        yaml.safe_dump(yaml_dict, f, sort_keys=False)

    print(f"\nWrote dataset YAML: {dataset_yaml_path}")
    print("YAML content:")
    print(yaml.safe_dump(yaml_dict, sort_keys=False))
    print("If class names are placeholders, edit the 'names' list in the YAML and save.")


No class names file found. Created placeholder names: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']
Wrote split files: D:\FYP\Datasets\NEW\train.txt (10204 lines), D:\FYP\Datasets\NEW\val.txt (2551 lines)

Wrote dataset YAML: D:\FYP\Datasets\NEW\dataset.yaml
YAML content:
train: D:\FYP\Datasets\NEW\train.txt
val: D:\FYP\Datasets\NEW\val.txt
nc: 5
names:
- class_0
- class_1
- class_2
- class_3
- class_4

If class names are placeholders, edit the 'names' list in the YAML and save.
